# Kaggle 雙 T4：架起自己的模型服務

YTP VocaBuddy 開發者之路・附錄 F・Lab 15（選修）配套 Notebook。每一格在做什麼、成功長什麼樣，請看教材：<https://tw-ytp.github.io/vocabuddy/#/lab15>

**開始前**：右側 Settings 的 Accelerator 選 **GPU T4 x2**、Internet 切到 **On**。Kaggle 不保留任何狀態，每次開機都要從 Cell 1 依序重跑；模型約 14 GB，第一次下載約五分鐘。

**本檔釘住的版本**（實測日期 2026-11-04；更新時請同步改教材附錄 B 與 F.1 的數據表）

| 項目 | 值 | 位置 |
|---|---|---|
| llama.cpp 預編譯版 | `b12063`（CUDA 12.8） | Cell 2 的 `TAG` |
| 模型 | Qwen3.8-27B Coder390 EfficientThink，Q2 量化、內建 MTP；視覺檔 Q8_0 | Cell 3 的 `REPO2` |
| 對外模型名稱 | `qwen3.8-27b` | Cell 4 的 `--alias` |
| 上下文 | 262144，`--fit off`，KV 快取 q8_0 | Cell 4 的 `start()` |
| 推測解碼 | `ngram-mod,draft-mtp`，MTP 一次猜 4 個 | Cell 4 的 `DEFAULT` |

**實測（暖機後、三個不同題目取中位數）**：寫新程式 35.0 tok/s、改既有檔案 109.4 tok/s、兩張卡各約 14.8 / 13.9 GB。

**提醒**：Cell 6 會印出 Base URL 與 API Key，等於你的 GPU 額度的鑰匙。分享或提交前先 **Restart & Run All** 確認可從頭跑完，再清除 Cell 6 的輸出；不要把含輸出的 Notebook 推上 GitHub。用完請按 **Stop Session**。

## Cell 1：確認兩張 GPU

應該看到兩列 `Tesla T4`，各 15360 MiB。只有一列或沒有，代表 Accelerator 沒選對。

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## Cell 2：安裝 llama.cpp

下載官方預編譯檔，不用編譯，約一分鐘。同一個工作階段重跑會跳過下載；連不上 github.com 通常是 Internet 沒開。最後一行印出 `llama-server` 的路徑就是成功。

In [ ]:
%%bash
set -e
TAG=b12063
cd /kaggle/working && mkdir -p llama

# 同一個 session 重跑時，已下載過就跳過
if find llama -name llama-server | grep -q .; then
  echo "已存在，略過下載"; find llama -name llama-server; exit 0
fi

# 連不上 GitHub 通常是 Settings → Internet 沒開
if ! curl -sSI --max-time 15 https://github.com > /dev/null; then
  echo "❌ 連不上 github.com：請到右側 Settings 把 Internet 打開（需已完成手機驗證），再重跑本格" >&2
  exit 1
fi

for f in llama-$TAG-bin-ubuntu-cuda-12.8-x64.tar.gz cudart-llama-$TAG-bin-ubuntu-cuda-12.8-x64.tar.gz; do
  curl -fsSL --retry 3 -o $f https://github.com/ggml-org/llama.cpp/releases/download/$TAG/$f   # 成功時不輸出，失敗才顯示原因
  tar -xzf $f -C llama && rm $f
done
find llama -name llama-server

## Cell 3：下載模型與視覺檔

主檔只含文字，看圖能力來自視覺檔。兩檔約 14 GB，放在 `/tmp/models`，約五分鐘。出現「unauthenticated requests」的警告可以忽略。

In [ ]:
from huggingface_hub import hf_hub_download

REPO2   = "nerkyor/Qwen3.8-27B-Coder390-EfficientThink-Opus5.5-GPT6Astra-Grok4.7-DSV4Pro-K3-SFT-RLOO-MTP-DFlash2"
MODEL2  = hf_hub_download(REPO2, "GGUF/Qwen3.8-27B-Coder390-EfficientThink-Q2LynnStyle-MTP.gguf", local_dir="/tmp/models")
MMPROJ2 = hf_hub_download(REPO2, "GGUF/mmproj-Qwen3.8-27B-Q8_0.gguf", local_dir="/tmp/models")
print(MODEL2, MMPROJ2, sep="\n")

## Cell 4：啟動伺服器

成功時印出「服務就緒」，兩張卡各佔約 14 到 15 GB。之後要重啟呼叫 `start(MODEL2, MMPROJ2, **DEFAULT)` 即可，金鑰不變；顯存不足時改 `start(MODEL2, MMPROJ2, ctx=131072)`。`spec` 可選 `none`、`mtp`、`ngram`、`hybrid`，Lab 15 的延伸步驟會用到。

In [ ]:
import os, glob, secrets, subprocess, time, requests

API_KEY = secrets.token_urlsafe(24)      # 服務要對外，所以一定要金鑰
PORT = 8000
LOG = None                               # 由 start() 依設定決定，每種設定各自一個 log 檔
DEFAULT = dict(spec="hybrid", draft_n=4) # 最終選定：ngram-mod + MTP

srv = glob.glob("/kaggle/working/llama/**/llama-server", recursive=True)[0]
libdirs = sorted({os.path.dirname(p) for p in
                  glob.glob("/kaggle/working/llama/**/*.so*", recursive=True)})
env = dict(os.environ, LD_LIBRARY_PATH=":".join(
    libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]))
server = globals().get("server")         # 重跑本格時，才關得掉上一次的 server

def wait_ready(proc, timeout=900):
    begin = time.time()
    while time.time() - begin < timeout:
        if proc.poll() is not None:
            raise RuntimeError(f"伺服器已退出，log 尾端：\n{open(LOG).read()[-3000:]}")
        try:
            if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=3).status_code == 200:
                print("服務就緒"); return
        except requests.RequestException:
            pass
        time.sleep(5)
    raise TimeoutError(f"逾時，請看 {LOG}")

def start(model, mmproj, spec="hybrid", ctx=262144, kv="q8_0", ub=1024, draft_n=4, extra=()):
    global server, LOG
    if server is not None and server.poll() is None:
        server.terminate(); server.wait()
    spec_type = {"none":   None,
                 "mtp":    "draft-mtp",
                 "ngram":  "ngram-mod",
                 "hybrid": "ngram-mod,draft-mtp"}[spec]       # --spec-type 可用逗號同時列出多種
    spec_args = [] if spec_type is None else ["--spec-type", spec_type]
    if spec_type and "mtp" in spec_type:                      # 只管 MTP；ngram-mod 用自己的 --spec-ngram-mod-n-*
        spec_args += ["--spec-draft-n-max", str(draft_n)]
    name = f"{spec}-n{draft_n}" if spec_type and "mtp" in spec_type else spec
    LOG = f"/kaggle/working/server_{name}_{kv}_{ctx//1024}k_ub{ub}{'_extra' if extra else ''}.log"
    with open(LOG, "a") as f:                                 # 附加模式：重啟多次，之前的錯誤也留得住
        f.write(f"\n===== {time.strftime('%m-%d %H:%M:%S')} start =====\n")
    server = subprocess.Popen([
        srv, "-m", model, "--mmproj", mmproj, "--alias", "qwen3.8-27b",
        "--jinja", "-ngl", "999", "-fa", "on", "-sm", "tensor", "-ts", "1,1",
        "--fit", "off",                                       # 不讓 llama.cpp 自動縮 ctx；放不下就直接報錯
        "-b", "4096", "-ub", str(ub),
        "--cache-type-k", kv, "--cache-type-v", kv,           # 量化 V cache 需要 -fa on
        *spec_args, *extra, "-c", str(ctx), "-np", "1",
        "--host", "127.0.0.1", "--port", str(PORT), "--api-key", API_KEY,
    ], stdout=open(LOG, "a"), stderr=subprocess.STDOUT, env=env, start_new_session=True)
    wait_ready(server)

start(MODEL2, MMPROJ2, **DEFAULT)
print("log:", LOG)
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv
!grep -iE "error|warn|ngram|mtp|draft" {LOG} | tail -20

## Cell 5：驗收測試

先暖機，再檢查五件事：文字回應、看圖、兩種速度（各三題取中位數）、兩張卡都有載入。約五分鐘。**五項都印「通過」才往下做**，並親自讀一遍第 2 題的完整回答。

In [ ]:
import base64, io, inspect, re, statistics, textwrap, shlex, heapq, fnmatch
from PIL import Image, ImageDraw, ImageFont

def chat(content, n=400):
    r = requests.post(
        f"http://127.0.0.1:{PORT}/v1/chat/completions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={"model": "qwen3.8-27b", "max_tokens": n, "temperature": 0,
              "chat_template_kwargs": {"enable_thinking": False},
              "messages": [{"role": "user", "content": content}]},
        timeout=900)
    if not r.ok:                                     # 顯示真正的錯誤，而不是 KeyError
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:1000]}")
    r = r.json()
    return r["choices"][0]["message"]["content"], r["timings"]

def timed(prompt, n=600):
    """回傳 (生成速度 tok/s, 推測解碼接受率)"""
    _, t = chat(prompt, n)
    acc = t.get("draft_n_accepted", 0) / t["draft_n"] if t.get("draft_n") else float("nan")
    return t["predicted_per_second"], acc

def speed(prompts, n=600):
    """每個 prompt 跑一次，回傳 (速度中位數, 接受率中位數, 每次的速度)"""
    xs = [timed(p, n) for p in prompts]
    return (statistics.median(x[0] for x in xs),
            statistics.median(x[1] for x in xs),
            [round(x[0], 1) for x in xs])

def edit_prompt(module, target):
    return (f"```python\n{inspect.getsource(module)}\n```\n在 {target} 開頭加上一行 "
            "logging.debug 記錄輸入參數，然後輸出修改後的完整檔案，不要省略任何部分。")

def warmup():
    """短、長 prompt 各跑一次，不計分；用的題目不在正式題目裡，避免 ngram 先「看過答案」"""
    timed("用 Python 寫一個二分搜尋函式並附上測試", 128)
    timed(edit_prompt(fnmatch, "fnmatch 函式"), 128)

# 每次換不同題目：同一題重複問、temperature=0 時輸出完全相同，ngram 會「抄上一次的答案」而虛高
NEW_PROMPTS = [
    "用 Python 寫一個完整的 LRU cache 類別並附上測試",
    "用 Python 寫一個完整的 Trie（字首樹）類別，支援 insert、search、starts_with，並附上測試",
    "用 Python 寫一個支援多執行緒的 token bucket 限流器類別，並附上測試",
]
EDIT_PROMPTS = [                                 # 三個不同的標準函式庫檔案
    edit_prompt(textwrap, "TextWrapper.wrap 方法"),
    edit_prompt(shlex,    "shlex.get_token 方法"),
    edit_prompt(heapq,    "heappush 函式"),
]

# 0 暖機
warmup()

# 1 文字
text, _ = chat("用一句話自我介紹", 200)

# 2 視覺：產生一張含錯誤訊息的圖片
img = Image.new("RGB", (1000, 240), "white")
ImageDraw.Draw(img).text((20, 20),
    'Traceback (most recent call last):\n  File "app.py", line 12, in <module>\n'
    '    total = price * qty\nTypeError: can\'t multiply sequence by non-int of type \'str\'',
    fill="black", font=ImageFont.load_default(size=24))
buf = io.BytesIO(); img.save(buf, format="PNG")
img_url = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
seen, _ = chat([{"type": "image_url", "image_url": {"url": img_url}},
                {"type": "text", "text": "這張截圖的錯誤是什麼？發生在哪個檔案的第幾行？怎麼修？"}], 800)

# 3 從零寫新程式的速度（3 個不同題目取中位數）
new_tps, new_acc, new_raw = speed(NEW_PROMPTS)

# 4 修改既有檔案的速度（3 個不同檔案取中位數；600 token 只夠輸出檔案前段，此題只量速度）
edit_tps, edit_acc, edit_raw = speed(EDIT_PROMPTS)

# 5 兩張卡的顯存
mem = [int(x) for x in subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.split()]

# 行號要完整比對，避免「120」「2012」也被當成 12
patterns = {"TypeError": r"TypeError", "app.py": r"app\.py",
            "12": r"(?:第\s*12\s*行|line\s*12\b|(?<!\d)12\s*行)"}
found = [k for k, p in patterns.items() if re.search(p, seen, re.I)]
checks = [
    ("文字回應",                    bool(text.strip()),         f"{len(text)} 字"),
    ("看圖讀出錯誤、檔名與行號",    len(found) == 3,            f"找到 {found}"),
    ("寫新程式 ≥ 12 tok/s",         new_tps >= 12,              f"{new_tps:.1f}（{new_raw}，接受率 {new_acc:.0%}）"),
    ("改既有檔案 ≥ 40 tok/s",       edit_tps >= 40,             f"{edit_tps:.1f}（{edit_raw}，接受率 {edit_acc:.0%}）"),
    ("兩張卡都載入 > 6 GB",         len(mem) >= 2 and all(m > 6000 for m in mem), f"{mem} MiB"),
]
for name, ok, detail in checks:
    print("通過" if ok else "失敗", "|", name, "|", detail)

print("\n===== 第 1 題完整回答 =====\n" + text)
print("\n===== 第 2 題完整回答 =====\n" + seen)

## Cell 6：開隧道，取得 Base URL 與 API Key

把印出的 **Base URL** 與 **API Key** 複製下來。這兩個值每次開機都不同；分享 Notebook 前請清除這一格的輸出。

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared && /kaggle/working/cloudflared --version

import re

CF_LOG = "/kaggle/working/cloudflared.log"
tunnel = subprocess.Popen(
    ["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate",
     "--url", f"http://127.0.0.1:{PORT}"],
    stdout=open(CF_LOG, "w"), stderr=subprocess.STDOUT,
    start_new_session=True)

public_url = None
for _ in range(30):
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open(CF_LOG).read())
    if m:
        public_url = m.group(0)
        break
    time.sleep(2)

print("Base URL:", public_url + "/v1" if public_url else f"尚未取得，請看 {CF_LOG}")
# ⚠ 公開網址：notebook 分享或設為 public 前，請清除這一格的輸出
print("API Key :", API_KEY)

## Cell 7：保活

最後執行。每五分鐘印一行，兩個都是 running 代表伺服器與隧道還活著。之後若要跑別的格子，先按 ■ 停止這一格，跑完再重新執行它；停止不會影響伺服器與隧道。

In [ ]:
try:
    while True:
        s, t = server.poll(), tunnel.poll()
        print(time.strftime("%H:%M"), "server", "running" if s is None else f"exited({s})",
              "| tunnel", "running" if t is None else f"exited({t})", flush=True)
        if s is not None:
            print(open(LOG).read()[-2000:]); break
        time.sleep(300)
except KeyboardInterrupt:
    print("停止監控（server 與 tunnel 仍在背景執行）")